# Create Multiscale Zarr Pyramid for Web Visualization

This notebook builds a multiscale Zarr pyramid from the MODIS snow phenology Icechunk store so the dataset can be visualized as a slippy web map using [zarr-layer](https://github.com/carbonplan/zarr-layer) without creating a separate visualization copy or running a tile server.

**Stack**
- [topozarr](https://github.com/carbonplan/topozarr): generates coarsened multi-resolution levels following the [GeoZarr multiscales spec](https://github.com/zarr-developers/geozarr-spec)
- [zarr-layer](https://github.com/carbonplan/zarr-layer): TypeScript library that fetches and renders Zarr as a custom MapLibre/Mapbox layer, reprojecting on the GPU on the fly

**Why multiscales?**  
The full store is 86 400 × 43 200 pixels. Without overview levels, zarr-layer would need to fetch the entire array at every zoom level. Multiscales let the viewer fetch only the appropriate coarsened level for the current viewport — qualitatively matching the performance of a traditional tile server.

**Projection note**  
The store is in MODIS Sinusoidal, *not* Web Mercator. zarr-layer reprojects on the GPU client-side, so we can write the pyramid in native projection and preserve pixel fidelity.

---
**Dependencies**: `topozarr` and `xproj` are added to `pixi.toml` as PyPI dependencies. Run `pixi install` before launching this notebook.

## Imports

In [ ]:
from pathlib import Path
import sys
import xarray as xr
import numpy as np
import zarr
import icechunk
import rioxarray
import xproj
from topozarr import create_pyramid, ZarrLayerVarConfig

sys.path.insert(0, str(Path('..').resolve()))
from modis_snow_phenology import Config

config = Config('config/config_v1.txt')

In [ ]:
import dask

# https://github.com/carbonplan/topozarr/blob/main/scripts/build_demo_data.py

# from https://github.com/carbonplan/ocr/blob/main/ocr/pipeline/create_pyramid.py
# zarr.config.set({'async.concurrency': 128})
# dask.config.set(scheduler='threads', num_workers=32)

# https://docs.dask.org/en/latest/scheduling.html
# from dask.distributed import Client
# client = Client() 

## 1. Open the Icechunk Store

In [ ]:
storage = icechunk.azure_storage(
    account=config.azure_storage_account,
    container=config.azure_container,
    prefix=config.icechunk_prefix,
    sas_token=config.azure_storage_sas_token,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session('main')

ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all',mask_and_scale=True)
ds

## 2. Prepare Dataset

topozarr needs:
1. A CRS assigned via `xproj` (`.proj.assign_crs()`)
2. The `spatial_ref` scalar coordinate removed — topozarr manages CRS metadata internally and the scalar causes issues during coarsening

Data comes out of the store as `float32` (Zarr's `mask_and_scale` replaces `int16` fill values with `NaN`). `create_pyramid(method='mean')` propagates NaNs correctly, so no extra masking is needed.

In [ ]:
# Extract CRS WKT from the rioxarray spatial_ref before we drop it
crs = ds.rio.crs
print('Native CRS:', crs)
print()
print('WKT:')
print(crs.to_wkt())

In [ ]:
# Drop the CF-convention spatial_ref scalar — xproj will carry the CRS instead
ds_clean = ds.drop_vars('spatial_ref')

# Assign CRS via xproj so topozarr can find it
ds_crs = ds_clean.proj.assign_crs(spatial_ref_crs={'wkt': crs.to_wkt()})

ds_crs

## 3. Create Multiscale Pyramid

### Choosing the number of levels

Each level coarsens by 2× in both spatial dimensions. The coarsest level (level 0 in topozarr's convention) should be small enough to fetch in a single request at the lowest zoom.

| Levels | Coarsest shape (y × x) | Coarsest pixel size |
|--------|------------------------|---------------------|
| 7      | 337 × 675              | ~59 km              |
| 8      | 169 × 337              | ~118 km             |
| 9      | 84 × 169               | ~236 km             |

8 levels gives a manageable coarsest tile (~169 × 337 pixels) while still preserving meaningful spatial detail at the finest level.

### Coarsening method
`method='mean'` is used for all three variables. For day-of-year metrics (SAD, SDD) this is a spatial average, which is visually reasonable. Override with `'min'` or `'max'` if you want conservative estimates.

### Non-spatial dimension
The `water_year` dimension is preserved at every pyramid level — zarr-layer handles it as a non-spatial dimension that the user can slice interactively.

In [ ]:
# Optional: supply colormap and range hints for zarr-layer
# SAD/SDD range is 1–366 (day of water year); max_consec is 0–366
layer_hints = {
    'SAD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='purples'),
    'SDD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='reds'),
    'max_consec_snow_days': ZarrLayerVarConfig(clim=[0, 366], colormap='blues'), 
}

In [ ]:
N_LEVELS = 5

pyramid = create_pyramid(
    ds_crs,
    levels=N_LEVELS,
    x_dim='x',
    y_dim='y',
    method='mean',
    target_chunk_bytes=int(0.5 * 1024 * 1024),  # 500 KB chunks — web-friendly
    chunks_per_shard=4,
    layer_hints=layer_hints,
)

pyramid

In [ ]:
for level in pyramid.dt:
    print(f"Level: {level}")
    print(pyramid.dt[level].ds.dims)
    print(f"variable SAD_DOWY encoding is: {pyramid.dt[level].ds['SAD_DOWY'].encoding}")
    print(f"variable SDD_DOWY encoding is: {pyramid.dt[level].ds['SDD_DOWY'].encoding}")
    print(f"variable max_consec_snow_days encoding is: {pyramid.dt[level].ds['max_consec_snow_days'].encoding}")

In [ ]:
# maybe here we can update the encoding?
int16_fill_value = np.iinfo(np.int16).min  # -32768
INT16_ENCODING = {
    'dtype': 'int16',
    '_FillValue': int16_fill_value,
    'write_empty_chunks': False,
}

# Merge with topozarr's chunk/shard encoding (which only has 'chunks'/'shards')
for level_path, level_enc in pyramid.encoding.items():
    for var_name in level_enc:
        level_enc[var_name].update(INT16_ENCODING)

## 4. Write Pyramid to Zarr

- **Azure**: write to the `uwcryo` storage account under a new prefix so zarr-layer can reach it from the browser

The DataTree hierarchy maps directly to Zarr group hierarchy:
```
modis_snow_phenology_multiscale.zarr/
├── .zattrs          ← multiscales + layer-hints metadata
├── 0/               ← coarsest level (169 × 337)
├── 1/
├── ...
└── 7/               ← native resolution (43200 × 86400)
```

### Write to Azure via Icechunk (recommended)

Use `icechunk.azure_storage` + `session.store` — the same pattern as the rest of the pipeline. This gives you a versioned, committable store on Azure that zarr-layer can read directly (the blog post confirms zarr-layer is compatible with Icechunk-backed stores).

For zarr-layer to reach the store from a browser the container must allow **anonymous blob reads** or use a SAS URL. Set the container access level to *Blob* in the Azure portal (or via `az storage container set-permission`).

In [3]:
MULTISCALE_PREFIX = 'modis_snow_phenology/modis_snow_phenology_multiscale_v1'

In [ ]:
# clean up any existing store at the target location before creating new one
remove_existing_store = True  # set to True to delete existing store and start fresh (warning: this will delete all existing data in the store!)
if remove_existing_store == True:
    import adlfs

    fs = adlfs.AzureBlobFileSystem(
        account_name=config.azure_storage_account,
        sas_token=config.azure_storage_sas_token,
    )
    prefix_path = f"{config.azure_container}/{MULTISCALE_PREFIX}"
    if fs.exists(prefix_path):
        fs.rm(prefix_path, recursive=True)
        print(f"Deleted existing store at {prefix_path}")
    else:
        print("No existing store found, nothing to delete")

In [ ]:
pyramid_storage = icechunk.azure_storage(
    account=config.azure_storage_account,
    container=config.azure_container,
    prefix=MULTISCALE_PREFIX,
    sas_token=config.azure_storage_sas_token,
)
pyramid_repo = icechunk.Repository.create(pyramid_storage)
session = pyramid_repo.writable_session('main')

pyramid.dt.to_zarr(session.store, mode='w', encoding=pyramid.encoding, zarr_format=3, consolidated=False)
session.commit('write multiscale pyramid: 5 levels, mean coarsening, WY2015-2024')

AZURE_URL = (
    f'https://{config.azure_storage_account}.blob.core.windows.net'
    f'/{config.azure_container}/{MULTISCALE_PREFIX}'
)
print(f'Written to: {AZURE_URL}')

## Code graveyard

In [ ]:
# # try example on topozarr README.md
# ds = xr.tutorial.open_dataset('air_temperature', chunks="auto", mask_and_scale=False) # mask_and_scale=False
# ds = ds.proj.assign_crs(spatial_ref="EPSG:4326")
# ds

# pyramid = create_pyramid(
#     ds,
#     levels=2,
#     x_dim="lon",
#     y_dim="lat",
#     method="mean",  # "mean" (default) | "max" | "min" | "sum"
# )
# print(pyramid) 

# print(pyramid.encoding)

# pyramid.dt['/1'].ds['air'].encoding

# encoding with mask_and_scale=True
# {'dtype': dtype('int16'),
#  'source': '/home/eric/.cache/xarray_tutorial_data/69c68be1605878a6c8efdd34d85b4ca1-air_temperature.nc',
#  'original_shape': (2920, 25, 53),
#  'scale_factor': np.float64(0.01)}

# encoding with mask_and_scale=False
# {'dtype': dtype('int16'),
#  'source': '/home/eric/.cache/xarray_tutorial_data/69c68be1605878a6c8efdd34d85b4ca1-air_temperature.nc',
#  'original_shape': (2920, 25, 53)}

# pyramid.dt['/1'].ds # this still looks lazy, that's good right???

# # access to original dataset in the datatree
# level_0_ds = pyramid.dt['/0'].ds.isel(time=0)
# level_1_ds = pyramid.dt['/1'].ds.isel(time=0)

# import matplotlib.pyplot as plt
# f,ax=plt.subplots(1,2, figsize=(12,6))
# level_0_ds.air.plot(ax=ax[0], vmin=200, vmax=300)
# ax[0].set_title('Level 0')
# level_1_ds.air.plot(ax=ax[1], vmin=200, vmax=300)
# ax[1].set_title('Level 1')
# f.tight_layout()

# for level in pyramid.dt:
#     print(f"Level: {level}")
#     print(pyramid.dt[level].ds.dims)
#     print(f"variable air's encoding is: {pyramid.dt[level].ds['air'].encoding}")
#     #print(pyramid.dt[level].ds)

In [ ]:
# write locally, prob bad idea
# LOCAL_OUT = Path('../data/modis_snow_phenology_multiscale.zarr')
# LOCAL_OUT.parent.mkdir(parents=True, exist_ok=True)

# # DataTree.to_zarr writes each level as a Zarr group.
# # Pass pyramid.encoding so topozarr's recommended chunk/shard sizes are respected.
# pyramid.dt.to_zarr(LOCAL_OUT, mode='w', encoding=pyramid.encoding)

# print(f'Written to {LOCAL_OUT.resolve()}')

# Quick verification — open the coarsest level
# store_check = zarr.open(LOCAL_OUT, mode='r')
# print(store_check.tree())